# Combined 3B — gold-contract diagnostic
Uses the same pinned Qwen2.5-3B-Instruct FP16 generator and combined repair loop as the completed selective-facts and tool runs. Supplies each task's gold contract to BOTH generation and verification. No intent-parser model, adapter, training or old result is required. Parser calls/tokens are zero. Reference programs remain evaluator-only.

Attach **stage2_combined_gold_source.zip**, enable **Internet**, select **T4 x2**, then **Save Version → Run All**. Run two smoke tasks, 20 fresh development tasks and 100 fresh diagnostic test tasks, each in separate output folders. Smoke acceptance is not a quality threshold. Download **stage2_combined_gold_3b.zip**, the summary ZIP and the executed notebook. Partial bundles are not completed results.

Optional: attach the completed `stage2_combined_3b_selective_proposed.zip` and `stage2_combined_3b_tools_proposed.zip` as SEPARATE datasets for a strict paired comparison in the final cell. Automatically extracted datasets are supported. They are not used during inference. Environment/settings mismatches stop comparison, while the completed gold bundle remains available. You can instead send back the gold bundle for local analysis.

This measures the combined effect of gold contracts on generation and checking/repair. Initial code can differ from predicted-contract runs. Gold is an oracle diagnostic, not a deployable system or a guaranteed upper bound; previously inspected tasks are not fresh held-out evidence. Total cost excludes parser inference in this run; generation/repair costs are separately reported.


In [ ]:
import datetime,hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR=WORK/('run_'+RUN_ID);RUN_DIR.mkdir()
DRIVE_RESULTS=None
def save_results():
    if DRIVE_RESULTS is not None:
        DRIVE_RESULTS.mkdir(parents=True,exist_ok=True)
        for p in RUN_DIR.iterdir():
            if p.is_file() and p.suffix in ['.zip','.log','.json','.txt']:shutil.copy2(p,DRIVE_RESULTS/p.name)
        print('Drive results:',DRIVE_RESULTS)
print('Run folder:',RUN_DIR)



In [ ]:
MANIFEST_SHA='6f38ed25cd58df3e2c83e543a3b707ef1a68c46f0d401553b50176e225393be4'
ROOT=WORK/('stage2_combined_gold_source_'+MANIFEST_SHA[:12])
def unpack(raw,read):
    assert hashlib.sha256(raw).hexdigest()==MANIFEST_SHA,'Wrong source ZIP version'
    manifest=json.loads(raw);assert manifest['experiment']=='stage2-combined-gold-3b-fp16-v1'
    for name,sha in manifest['sha256'].items():
        relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
        data=read(name);assert hashlib.sha256(data).hexdigest()==sha,f'Changed source/input: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==data,f'Changed working source: {name}'
        else:path.write_bytes(data)
    sidecar=ROOT/'_stage2_combined_gold_manifest.json'
    if sidecar.exists():assert sidecar.read_bytes()==raw,'Changed source manifest'
    else:sidecar.write_bytes(raw)
    print('Verified fresh-run source/data:',len(manifest['sha256']),'files',MANIFEST_SHA)
found=False
for path in INPUT.rglob('manifest.json'):
    if hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        unpack(path.read_bytes(),lambda name:(path.parent/name).read_bytes());found=True;break
if not found:
    for path in INPUT.rglob('stage2_combined_gold_source.zip'):
        with zipfile.ZipFile(path) as z:
            assert len(z.namelist())==len(set(z.namelist()))
            prefix='stage2_combined_gold_source/';name=prefix+'manifest.json'
            if name not in z.namelist():continue
            raw=z.read(name)
            if hashlib.sha256(raw).hexdigest()!=MANIFEST_SHA:continue
            unpack(raw,lambda name:z.read(prefix+name));found=True;break
assert found,'Attach/place stage2_combined_gold_source.zip in the input folder.'
PACKAGES=WORK/'stage2_combined_gold_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.is_available(); assert all(torch.cuda.get_device_properties(i).total_memory/2**30>=12 for i in range(torch.cuda.device_count())); print(torch.__version__,[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','eval.stage2_combined_3b.preflight'],cwd=ROOT,env=ENV,check=True)


In [ ]:
DEV_COUNT=20
def execute(smoke,split='test'):
    name='stage2_combined_gold_3b'+('_smoke' if smoke else '_dev' if split=='dev' else '')
    out=RUN_DIR/name;assert not out.exists(),'Use a fresh run folder.'
    command=[sys.executable,'-m','eval.stage2_combined_gold.runner','--method','gold','--preset','gold_contract','--out',str(out),'--packages',str(PACKAGES),'--split',split]
    if smoke:command+=['--smoke']
    log=RUN_DIR/(name+'.log')
    try:
        with log.open('w') as f:
            process=subprocess.Popen(command,cwd=ROOT,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in process.stdout:
                print(line,end='');f.write(line);f.flush()
            code=process.wait()
    finally:save_results()
    assert code==0,f'Run failed: inspect {log} and the partial ZIP. A partial job is not a result.'
    assert json.loads((out/'completion.json').read_text())['complete']
    assert len(list((out/'tasks').glob('*.json')))==(2 if smoke else DEV_COUNT if split=='dev' else 100)
    return out.with_suffix('.zip')
SMOKE=execute(True)
DEV=execute(False,split='dev')
print('Development summary:',(DEV.with_suffix('')/'summary.json').read_text())
print('Smoke and end-to-end development completed. Inspect the development summary; the next cell runs 100 diagnostic test tasks freshly.')


In [ ]:
BUNDLE=execute(False)
OUT=RUN_DIR/'stage2_combined_gold_summary'
subprocess.run([sys.executable,'-m','eval.stage2_combined_gold.collect','--gold',str(BUNDLE),'--out',str(OUT)],cwd=ROOT,env=ENV,check=True)
from IPython.display import display,FileLink,Markdown
display(Markdown((OUT/'SUMMARY.md').read_text()))
display(FileLink(str(BUNDLE)));display(FileLink(str(OUT.with_suffix('.zip'))))
print('Download the gold bundle and this executed notebook. Predicted-contract outputs are optional in the next cell.')


In [ ]:
# Optional: attach the two COMPLETED combined result bundles to compare task by task.
# This cell skips comparison when neither result is attached. The gold run is independent.
paths=[]
for method in ['selective','tools']:
    name='stage2_combined_3b_'+method+'_proposed.zip'
    candidates=list(INPUT.rglob(name))
    # Support Kaggle auto-extracted datasets as well as intact ZIPs.
    for path in INPUT.rglob('plan.json'):
        try:
            plan=json.loads(path.read_text())
            if plan.get('experiment')!='stage2-combined-3b-fp16-v1' or plan.get('method')!=method or plan.get('smoke') or plan.get('split')!='test':continue
            folder=path.parent;seal=json.loads((folder/'bundle.json').read_text())
        except (OSError,ValueError):continue
        assert seal['complete'],'Incomplete predicted run'
        local=RUN_DIR/('predicted_'+method+'_'+str(len(candidates))+'.zip')
        with zipfile.ZipFile(local,'w',zipfile.ZIP_DEFLATED) as z:
            for name,digest in seal['sha256'].items():
                relative=Path(name);assert not relative.is_absolute() and '..' not in relative.parts
                data=(folder/relative).read_bytes();assert hashlib.sha256(data).hexdigest()==digest
                z.writestr(name,data)
            z.writestr('bundle.json',json.dumps(seal))
        candidates.append(local)
    distinct={}
    for path in candidates:
        with zipfile.ZipFile(path) as z:seal=json.loads(z.read('bundle.json'))
        distinct[json.dumps(seal,sort_keys=True)]=path
    assert len(distinct)<=1,f'Multiple different full results for {method}'
    paths.extend(distinct.values())
if paths:
    OUT=RUN_DIR/'stage2_combined_gold_comparison'
    subprocess.run([sys.executable,'-m','eval.stage2_combined_gold.collect','--gold',str(BUNDLE),'--predicted',*map(str,paths),'--out',str(OUT)],cwd=ROOT,env=ENV,check=True)
    display(Markdown((OUT/'SUMMARY.md').read_text()));display(FileLink(str(OUT.with_suffix('.zip'))))
else:
    print('No predicted bundles attached. Gold results are complete; upload them for local comparison.')
